# Automating a Web Pen-Test on Real Targets: Chains → RAG → Agents + MCP
### A Concrete, Real-Data Walkthrough of the Three Parts

**Audience:** Graduate students who have completed the LangChain, RAG, and MCP modules.

**Goal:** Re-run the same three-stage story as `part2_orchestration_simulated.ipynb`, but on **real, authorized vulnerable websites** instead of a simulated `nmap`. You will see genuine HTTP recon data flow through each stage:

1. **Chains** — `web recon → LLM parses → LLM analyzes → report`. *The LLM guesses, with no OWASP references.*
2. **RAG** — `… → retrieve web-security guidance → analyze → report`. *Answers grounded in a real knowledge base, citing OWASP categories.*
3. **Agent + MCP** — *the agent decides which recon tool to call*, through a standard `recon_server_web.py`.

> **Ethics & authorization.** Recon here targets only **authorized** sites: **`scanme.nmap.org`** (published by the Nmap Project expressly so people can test scanning tools) and the **Acunetix `vulnweb.com` family** (`testphp.vulnweb.com`, …). Every call makes at most **one ordinary HTTP GET** — no scanning, fuzzing, or exploitation. **Never point these tools at a site you are not explicitly authorized to test.**

> **Reproducibility.** If the network is blocked, recon falls back to a clearly-labelled cached sample, so every cell still runs. Stages 1–2 and the offline agent need **no API key**; the **live agent** needs `OPENAI_API_KEY` and the `mcp` SDK.

> **This vs. the VM project.** This notebook is the *worked example* over web targets. Your **project** is to stand up a local exploitable VM (e.g., Metasploitable), run `nmap` and other tools against it, and adapt these three stages — see the closing cell.

## Setup

In [ ]:
# (Install once if missing:)
# !pip install -q langchain-core sentence-transformers
# Optional, only for the Stage 3 LIVE agent:
# !pip install -q mcp langchain-mcp-adapters langgraph langchain-openai

import re                      # `re` = regular expressions; used later to pull "A05:2021" OWASP tags out of text
import ssl                     # `ssl` = builds a TLS context so urllib can verify HTTPS certificates
import urllib.request          # `urllib.request` = Python's built-in HTTP client; we use it for ONE GET per target
from typing import List        # `List` = a type hint (e.g. List[str]) documenting "a list of strings"

# LangChain "core" primitives — the small building blocks from the LangChain module:
from langchain_core.runnables import RunnableLambda          # wraps a plain Python function so it can sit inside a chain
from langchain_core.output_parsers import StrOutputParser    # takes the model's message object and returns just its text
from langchain_core.prompts import ChatPromptTemplate        # builds a (system+user) prompt from a template with {slots}
from langchain_core.messages import AIMessage                # the object an LLM returns; our offline stand-in returns one too

print("Core imports ready.")   # a simple confirmation that the cell ran

### Real (but safe) web recon, with an offline cache

We only contact **authorized** targets, we make a single GET per call, and if the network is unavailable we fall back to a **cached sample** so the part is reproducible. This is the same recon logic the `recon_server_web.py` exposes as MCP tools in Stage 3.

In [ ]:
# ── Authorized targets ───────────────────────────────────────────────────────
# ALLOWED_HOSTS: a SET of hostnames we are permitted to contact. It acts as an
# allowlist — the single most important safety control in this notebook.
ALLOWED_HOSTS = {"scanme.nmap.org",         # published by the Nmap Project for testing scanning tools
                 "testphp.vulnweb.com",     # Acunetix deliberately-vulnerable demo (nginx/PHP)
                 "testasp.vulnweb.com",     # Acunetix demo (IIS/ASP)
                 "testaspnet.vulnweb.com",  # Acunetix demo (ASP.NET)
                 "testhtml5.vulnweb.com"}   # Acunetix demo (HTML5)

# SECURITY_HEADERS: the standard HTTP response headers a hardened site SHOULD send.
# We will later flag any of these that are MISSING from a target's response.
SECURITY_HEADERS = ["Strict-Transport-Security",   # forces HTTPS (HSTS)
                    "Content-Security-Policy",      # restricts which scripts/resources may load (anti-XSS)
                    "X-Frame-Options",              # blocks framing of the page (anti-clickjacking)
                    "X-Content-Type-Options",       # stops the browser MIME-sniffing the body
                    "Referrer-Policy"]              # controls how much referrer info leaks to other sites

# CACHED: a fallback table of realistic responses, keyed by URL. If the LIVE request
# fails (e.g. the classroom blocks outbound traffic) we use these so every cell still
# runs. `status` = the HTTP status code; `headers` = a dict of response headers.
CACHED = {
    "http://scanme.nmap.org": {"status": 200, "headers": {
        "Server": "Apache/2.4.7 (Ubuntu)", "Content-Type": "text/html", "Connection": "close"}},
    "http://testphp.vulnweb.com": {"status": 200, "headers": {
        "Server": "nginx/1.19.0", "Content-Type": "text/html; charset=UTF-8",
        "Connection": "close", "X-Powered-By": "PHP/5.6.40"}},
}

def host_of(url):
    # Extract just the hostname from a full URL string. Step by step:
    #   url.split("://", 1)[-1]  -> drop the "http://" scheme, leaving "host/path..."
    #   .split("/", 1)[0]        -> keep everything before the first "/", i.e. "host[:port]"
    #   .split(":", 1)[0]        -> drop any ":port", leaving the bare hostname
    return url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0]

def http_recon(url, timeout=8):
    # Perform ONE ordinary HTTP GET against `url` and return a dict describing the response.
    # `timeout` = seconds to wait before giving up on the connection.
    if host_of(url) not in ALLOWED_HOSTS:                  # SAFETY GATE: refuse anything not on the allowlist
        raise ValueError(f"Refused: {host_of(url)} is not authorized.")
    try:
        # `req` = a request object carrying a polite, identifying User-Agent header.
        req = urllib.request.Request(url, headers={"User-Agent": "edu-web-recon/1.0"})
        # Open the connection (verifying TLS via a default ssl context); `r` = the response object.
        with urllib.request.urlopen(req, timeout=timeout, context=ssl.create_default_context()) as r:
            # r.status = HTTP code (200, 404, ...); dict(r.headers) = all response headers as a dict.
            return {"status": r.status, "headers": dict(r.headers), "live": True, "url": url}
    except Exception as e:                                  # any failure (offline, DNS, refused) lands here
        c = CACHED.get(url, {"status": None, "headers": {}})   # `c` = a cached sample for this URL (or an empty one)
        # Return the cached data, marked live=False, and keep the error text for transparency.
        return {"status": c["status"], "headers": dict(c["headers"]),
                "live": False, "url": url, "error": str(e)}

# TARGET: the site this notebook assesses by default. scanme.nmap.org is authorized by
# the Nmap Project. Change it to another ALLOWED host to assess that one instead.
TARGET = "http://scanme.nmap.org"
recon = http_recon(TARGET)            # `recon` = the response dict for TARGET (from the live server or the cache)

# Print a short summary so you can see what came back:
print(f"Recon of {TARGET}  ({'LIVE' if recon['live'] else 'CACHED'}):")
print("  HTTP", recon["status"])      # the HTTP status code
for k, v in recon["headers"].items(): # iterate header name (k) / value (v) pairs
    print(f"  {k}: {v}")

Turn the raw response into a list of **findings** — the web-recon analog of nmap's service list. Each finding is a missing security header, a version disclosure, or cleartext HTTP.

In [ ]:
def analyze_findings(recon: dict) -> List[str]:
    # Turn a recon response dict into a list of human-readable "findings".
    # `recon` = the dict returned by http_recon().
    headers = recon["headers"]                    # `headers` = the response headers dict
    present = {k.lower() for k in headers}        # `present` = header names lowercased, for case-insensitive checks
    findings = []                                 # `findings` = the list of issues we build up and return

    if recon["url"].lower().startswith("http://"):   # if the URL is plain HTTP (not HTTPS)...
        findings.append("Cleartext HTTP (no TLS)")   # ...record that traffic is unencrypted

    for h in SECURITY_HEADERS:                       # check each expected security header
        if h.lower() not in present:                 # if the server did NOT send it...
            findings.append(f"Missing security header: {h}")   # ...record it as missing

    if "server" in present:                          # the `Server` header reveals the web-server product/version
        findings.append(f"Server version disclosure: {headers.get('Server')}")

    xpb = headers.get("X-Powered-By")                # `xpb` = the X-Powered-By header value (or None if absent)
    if xpb:                                          # if present, it leaks the app technology/version
        eol = " (PHP 5.x is end of life)" if "php/5" in xpb.lower() else ""   # flag the well-known end-of-life case
        findings.append(f"Technology disclosure: {xpb}{eol}")

    return findings                                  # hand back the complete list of findings

findings = analyze_findings(recon)   # `findings` = the list of issues found for TARGET
print("Findings:")
for f in findings:                   # print each finding on its own line
    print("  -", f)

---
## STAGE 1 — Chains: An LLM-Powered Web-Recon Assistant

*Theme: “Use LangChain to turn raw recon into a readable report.”*

# Stage 1 — Recon → Parse → Analyze → Report (a Chain)

- Fetch real HTTP headers from an authorized target
- Extract a list of **findings** (missing headers, disclosures)
- LLM **analyzes** each finding for likely risk
- LLM **generates** a readable report
- Wired as a LangChain chain: `prompt | model | parser`

> ### Deeper Explanation
>
> We start exactly where the simulated capstone did, but with real data. We fetch the actual HTTP headers of an authorized vulnerable site, turn them into a list of findings, and feed those to an LLM acting as a security analyst, all strung together as a LangChain chain. The shape is the familiar prompt-model-parser you learned in the LangChain module, now applied to genuine recon output. We use the offline stand-in model so it runs in class; the live version is a one-line swap shown later. Notice this is already useful — in minutes you have an assistant that turns a raw header dump into prose.

In [ ]:
def _analyst_from_memory(prompt_value):
    # An OFFLINE stand-in for an LLM. A real model would reason here; this stub just
    # restates the findings so the pipeline runs with NO API key. The point of Stage 1
    # is that the model "guesses", so this stub deliberately produces vague output.
    # `prompt_value` = the filled-in prompt object LangChain passes to the model.
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)  # prompt as plain text
    # `items` = the bullet lines (those starting with "-") pulled back out of the prompt text:
    items = [l.strip("- ").strip() for l in text.splitlines() if l.strip().startswith("-")]
    body = ("Based on general knowledge, these findings *may* indicate weaknesses "
            "(UNVERIFIED — the model is guessing, with no references):\n")   # `body` = the report text we build up
    for f in items:                              # add one vague line per finding (a guess, by design)
        body += f"  - {f}: probably worth hardening; could enable some attack.\n"
    return AIMessage(content=body + "Recommend manual verification.")   # return it as an AIMessage, like a real model

analyst_memory_llm = RunnableLambda(_analyst_from_memory)   # wrap the function so it can act as a step in a chain

# `report_prompt` = a reusable prompt with a system role (instructions) and a user role
# containing a {findings} slot that gets filled in at run time.
report_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a web-application security assistant. Analyze the findings and write a "
               "short report."),
    ("user", "Findings:\n{findings}"),
])

# `stage1_chain` = the chain, built with the LCEL pipe `|`: prompt -> model -> text parser.
stage1_chain = report_prompt | analyst_memory_llm | StrOutputParser()

# .invoke() fills {findings} with our bullet list and runs the whole chain; `report` = the final string.
report = stage1_chain.invoke({"findings": "\n".join(f"- {f}" for f in findings)})
print("=== STAGE 1 REPORT (chain, LLM from memory) ===\n")
print(report)

# Stage 1 — The Critical Limitation

- The LLM **guesses** from training memory
- No OWASP category, no CWE, no concrete remediation
- Cannot tie a finding to an authoritative standard
- → **hallucination risk** in a security context = dangerous
- This motivates Stage 2 (RAG)

> ### Deeper Explanation
>
> Now the teaching moment. Re-read that report: it is fluent and plausible, but it cites nothing. The model is guessing from general training, so it gives no OWASP category, no CWE identifier, and no standards-based remediation a real assessment needs. In security, a vague or wrong answer is dangerous — a hallucinated 'looks fine' can leave a real hole open. The lesson is that an LLM's memory is the wrong source of truth for findings that should map to authoritative guidance. That is exactly what Retrieval-Augmented Generation fixes, which is Stage 2.

In [ ]:
# This cell simply prints, in plain language, WHY the Stage 1 output is not trustworthy.
print("What Stage 1 produced:")
print("  - source of 'facts' : the LLM's training memory (a guess)")
print("  - OWASP / CWE refs   : none")
print("  - remediation        : vague")
print("  - trustworthy        : NOT for a real assessment")
print("\nFix: ground each finding in a real web-security knowledge base -> Stage 2 (RAG).")

---
## STAGE 2 — RAG: Ground Each Finding in OWASP Guidance

*Theme: “From guessing → retrieving real security guidance.”*

# Stage 2 — Retrieve Real Guidance, Then Analyze

- Embed `web_security_kb.md` into a vector store (the RAG module)
- For each finding, **retrieve** the matching OWASP guidance
- Feed the *retrieved* guidance to the LLM as context
- Report now cites **OWASP categories** and concrete fixes
- Pipeline: `recon → findings → retrieve → analyze → report`

> ### Deeper Explanation
>
> Stage 2 upgrades the assistant exactly as the RAG part promised: we replace the model's memory with retrieval from a real knowledge base. Using the embeddings and vector-store tools from the RAG module, we index `web_security_kb.md`, then for each finding we retrieve the most relevant guidance and pass it into the prompt as context. The model's job changes from recall to reading comprehension over supplied facts. The report now references concrete OWASP categories like A05 Security Misconfiguration with real remediation. Same recon, same model, but grounded output.

In [ ]:
from sentence_transformers import SentenceTransformer       # local embedding model (text -> vectors); no API needed
from langchain_core.embeddings import Embeddings               # base class so our wrapper fits LangChain's vector store
from langchain_core.documents import Document                  # a small container: page_content (text) + metadata
from langchain_core.vectorstores import InMemoryVectorStore    # a simple, RAM-only vector database

class STEmbeddings(Embeddings):
    # Adapter that lets a sentence-transformers model be used as a LangChain "Embeddings".
    def __init__(self, name="all-MiniLM-L6-v2"):       # `name` = which pretrained model to load (small & fast)
        self.m = SentenceTransformer(name)             # `self.m` = the loaded model (downloaded & cached on first use)
    def embed_documents(self, texts):                  # embed a LIST of texts -> list of vectors
        return self.m.encode(texts, normalize_embeddings=True).tolist()   # normalize so cosine similarity behaves well
    def embed_query(self, text):                       # embed a SINGLE query string -> one vector
        return self.m.encode(text, normalize_embeddings=True).tolist()

kb_text = open("web_security_kb.md", encoding="utf8").read()   # `kb_text` = the whole knowledge-base file as one string
# `sections` = the file split on "#" headers; each non-empty chunk is one guidance entry.
sections = [s.strip() for s in kb_text.split("#") if s.strip()]
# `docs` = each section wrapped in a Document, recording its title (first line) as metadata.
docs = [Document(page_content=s, metadata={"title": s.splitlines()[0]}) for s in sections]

kb_store = InMemoryVectorStore(STEmbeddings())   # `kb_store` = the vector DB, using our embedding wrapper
kb_store.add_documents(docs)                     # embed every section and index it for similarity search
print(f"Indexed {len(docs)} guidance entries into the vector store.\n")

def retrieve_guidance(finding: str, k: int = 1, threshold: float = 0.30) -> str:
    # Find the KB entry most relevant to `finding`.
    # `k` = how many top matches to consider; `threshold` = minimum similarity score to accept.
    hits = kb_store.similarity_search_with_score(finding, k=k)            # `hits` = list of (Document, score) pairs
    kept = [d.page_content for d, score in hits if score >= threshold]    # keep only confident matches
    return kept[0] if kept else f"No guidance found for: {finding}"       # return the best text, or a "none" message

for f in findings:   # show, per finding, which guidance entry was retrieved (just its title line)
    print(f"  {f[:42]:44s} ->  {retrieve_guidance(f).splitlines()[0]}")

Now build the **RAG chain**: retrieve guidance for the findings, then have the (stand-in) LLM write a grounded report that cites the OWASP categories from the retrieved context.

In [ ]:
def _analyst_grounded(prompt_value):
    # OFFLINE stand-in that answers ONLY from the retrieved guidance (extractive — no guessing).
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)   # prompt as text
    # `context` = the retrieved-guidance portion of the prompt (after our marker, before the user turn):
    context = text.split("# Retrieved guidance", 1)[-1].split("Human:", 1)[0].strip()
    # `owasp` = the unique OWASP category tags (e.g. "A05:2021 ...") found in that context, via regex:
    owasp = sorted(set(re.findall(r"A0\d:2021[^.\n]*", context)))
    head = f"GROUNDED REPORT — cites {len(owasp)} OWASP categor(ies): {'; '.join(owasp)}\n\n"   # `head` = a summary line
    return AIMessage(content=head + context[:700] + (" ..." if len(context) > 700 else ""))     # header + trimmed context

grounded_llm = RunnableLambda(_analyst_grounded)   # wrap as a chain step

# `rag_prompt` = a prompt that injects the retrieved {context} into the system instructions.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a web-app security assistant. Use ONLY the retrieved guidance below.\n"
               "# Retrieved guidance\n{context}"),
    ("user", "Assess {url} given these findings:\n{findings}"),
])

def build_context(_):   # `_` = the chain input (ignored here); we always assess the current `findings`
    return "\n\n".join(retrieve_guidance(f) for f in findings)   # join the guidance retrieved for every finding

# `stage2_chain` = the RAG chain. The opening dict prepares the three prompt slots in parallel:
#   context  -> retrieved KB text (build_context)
#   findings -> the bullet list of findings
#   url      -> the target URL, passed straight through
stage2_chain = (
    {"context": RunnableLambda(build_context),
     "findings": lambda _: "\n".join(f"- {f}" for f in findings),
     "url": RunnableLambda(lambda u: u)}
    | rag_prompt | grounded_llm | StrOutputParser()   # then: fill the prompt -> grounded model -> plain text
)

print("=== STAGE 2 REPORT (RAG, grounded in OWASP guidance) ===\n")
print(stage2_chain.invoke(TARGET))   # run it; the input (TARGET) flows into the `url` slot

# Stage 2 — What Improved

- Findings mapped to **real OWASP categories**, not memory
- Report gives **concrete, standards-based** remediation
- Hallucination sharply **reduced**
- Knowledge base is **updatable** without retraining
- Still a *fixed* pipeline — Stage 3 makes it dynamic

> ### Deeper Explanation
>
> Compare the two reports and the win is obvious: Stage 2 maps each finding to a real OWASP category with standards-based remediation drawn from documents, while Stage 1 offered fluent guesses. Accuracy is up, hallucination is down, and the knowledge base can be refreshed the moment guidance changes — no retraining. But notice what is still true: the pipeline is fixed. We hard-coded recon, then findings, then retrieve, then analyze. What if a finding should trigger a deeper look, or fetching robots.txt? For that the system must decide for itself — Stage 3: agents and MCP.

In [ ]:
# Print a side-by-side comparison of Stage 1 vs Stage 2 across a few dimensions.
print(f"{'':22s}{'STAGE 1 (chain)':22s}{'STAGE 2 (RAG)'}")   # the column headers
print("-" * 62)
# Each tuple below = (row label, Stage 1 value, Stage 2 value):
for label, a, b in [("Source of facts",  "LLM memory", "web_security_kb.md"),
                    ("OWASP references", "none",       "yes (e.g. A05:2021)"),
                    ("Remediation",      "vague",      "concrete, standards-based"),
                    ("Hallucination",    "high",       "reduced"),
                    ("Updatable",        "retrain",    "edit the KB")]:
    print(f"{label:22s}{a:22s}{b}")   # left-pad each column to 22 chars so they line up

---
## STAGE 3 — Agents + MCP: Let the System Decide

*Theme: “From pipelines → autonomous web-recon systems.”*

# Stage 3 — An Agent Orchestrates MCP Recon Tools

- `recon_server_web.py` exposes `fetch_headers`, `check_security_headers`, `get_robots`, `lookup_guidance`
- An **agent** decides *which* tool to call, *when*, and *with what*
- Loop: reason → call a tool → observe → repeat → report
- Tools are standardized & reusable (swap/add servers freely)
- Final pipeline is **dynamic**, not hard-coded

> ### Deeper Explanation
>
> Stage 3 removes the last constraint: the fixed sequence. Instead of us scripting recon-then-retrieve, we expose those capabilities as MCP tools on `recon_server_web.py` and hand them to an agent. The agent runs the reason-act-observe loop: it decides to fetch the headers, reads them, decides to check which security headers are missing, looks up guidance for each, and writes the report. The control flow now comes from the model, and because the tools are MCP-standardized we could add a TLS-checker or a CVE server later without touching the agent. First a deterministic offline version with no key; then the live LangGraph + MCP agent.

In [ ]:
# ── Offline agent: a deterministic loop that DECIDES its next tool call ──────
# These three functions are the "tools" the agent may call (the same recon logic as above):
def tool_fetch_headers(url): return http_recon(url)                            # -> the raw response dict
def tool_check_security_headers(url): return analyze_findings(http_recon(url)) # -> the list of findings
def tool_lookup_guidance(finding): return retrieve_guidance(finding).splitlines()[0]  # -> a guidance title line

# `AGENT_TOOLS` = a name -> function registry, so the loop can call a tool by its string name.
AGENT_TOOLS = {"fetch_headers": tool_fetch_headers,
               "check_security_headers": tool_check_security_headers,
               "lookup_guidance": tool_lookup_guidance}

def offline_agent(goal_url, max_steps=10):
    # Simulate an agent's reason -> act -> observe loop WITHOUT a real LLM.
    # `goal_url` = the site to assess; `max_steps` = a safety cap on the number of iterations.
    print(f"GOAL: assess {goal_url}\n")
    fetched = False          # `fetched` = have we run the header check yet?
    findings_left = []       # `findings_left` = findings still needing a guidance lookup
    report = []              # `report` = the guidance results gathered so far
    for step in range(1, max_steps + 1):   # iterate steps 1..max_steps
        # --- the "policy": rules that pick the next action (a real agent lets the LLM choose) ---
        if not fetched:                                  # first action: always gather the findings
            action, args = "check_security_headers", {"url": goal_url}
        elif findings_left:                              # next: look up guidance for each finding in turn
            action, args = "lookup_guidance", {"finding": findings_left.pop(0)}   # pop(0) = take the next finding
        else:                                            # nothing left to do -> stop the loop
            print(f"  step {step}: REASON -> enough info, write the report"); break
        print(f"  step {step}: REASON -> call {action}({args})")   # show the decision the agent made
        result = AGENT_TOOLS[action](**args)             # ACT: call the chosen tool with its arguments
        shown = result if isinstance(result, str) else "; ".join(result)   # `shown` = result formatted for printing
        print(f"           OBSERVE -> {shown[:72]}{'...' if len(shown) > 72 else ''}")   # show a short preview
        if action == "check_security_headers":           # if we just fetched the findings...
            fetched = True; findings_left = list(result) # ...remember them for the lookups that follow
        else:
            report.append(result)                        # otherwise store this guidance result
    print("\n=== STAGE 3 REPORT (agent-orchestrated) ===")
    for r in report:
        print("  -", r)

offline_agent(TARGET)   # run the loop against the default target

# Stage 3 (Live) — LangGraph Agent over the MCP Server

- `load_mcp_tools` → MCP tools become LangChain tools
- `create_react_agent(llm, tools)` builds the agent loop for you
- The LLM picks recon tools dynamically at runtime
- Needs `OPENAI_API_KEY` + the `mcp` SDK (else this step skips)
- Connects to `recon_server_web.py` over stdio

> ### Deeper Explanation
>
> Finally, the real thing. We connect to `recon_server_web.py` over stdio, load its tools as LangChain tools with `load_mcp_tools`, and build a LangGraph ReAct agent with `create_react_agent`, backed by a real OpenAI model. Now the agent genuinely decides: it calls check_security_headers on the authorized target, looks up guidance for each finding, and synthesizes a grounded report — and it would just as easily use a new tool if we added one to the server. This step needs an API key and the `mcp` package, so it is guarded; without them it prints how to run it. Everything before this ran offline, so the part works either way.

In [ ]:
import os, sys   # `os` = read the OPENAI_API_KEY env var; `sys` = sys.executable is the path to THIS Python

async def run_live_agent(goal):
    # Connect to the MCP server, load its tools, and let a REAL LLM agent use them.
    # `goal` = the natural-language task we hand the agent.
    from mcp import ClientSession, StdioServerParameters    # MCP client session + how to launch a stdio server
    from mcp.client.stdio import stdio_client               # context manager that spawns the server over stdio
    from langchain_mcp_adapters.tools import load_mcp_tools  # converts MCP tools -> LangChain tools
    from langgraph.prebuilt import create_react_agent        # builds a ready-made reason+act agent loop
    from langchain_openai import ChatOpenAI                  # the OpenAI chat-model wrapper
    import asyncio                                           # used for the wait_for() timeouts below

    # `errlog` -> a real file. Passing Jupyter's sys.stderr would crash (it has no fileno);
    # discarding stderr instead would hide a dead server and make initialize() hang.
    errlog = open("web_recon_server_stderr.log", "w")
    # `params` = how to launch the server: run THIS python on recon_server_web.py over stdio.
    params = StdioServerParameters(command=sys.executable, args=["recon_server_web.py"])
    async with stdio_client(params, errlog=errlog) as (read, write):   # spawn server; `read`/`write` = its stdio pipes
        async with ClientSession(read, write) as session:             # `session` = the MCP conversation object
            await asyncio.wait_for(session.initialize(), timeout=30)  # MCP handshake (fail at 30s rather than hang)
            tools = await load_mcp_tools(session)                     # `tools` = the server's tools, as LangChain tools
            # `agent` = a ReAct agent driven by a deterministic (temperature=0) GPT model plus those tools.
            agent = create_react_agent(ChatOpenAI(model="gpt-4o-mini", temperature=0), tools)
            result = await asyncio.wait_for(                          # run the agent on our goal (cap at 120s)
                agent.ainvoke({"messages": [("user", goal)]}), timeout=120)
            return result["messages"][-1].content                    # the LAST message = the agent's final answer

def have(mod):
    # Return True if a module is installed, so we can skip the live step gracefully if not.
    import importlib.util
    return importlib.util.find_spec(mod) is not None   # find_spec returns None when the module is absent

def run_async(coro):
    # Run an async coroutine `coro` from anywhere, including a Jupyter cell.
    # A Jupyter cell ALREADY runs an event loop, and asyncio.run() refuses a nested one.
    # nest_asyncio is NOT used: on Windows, spawning the stdio server needs a ProactorEventLoop
    # that a re-entrant loop can't drive. So we run the coroutine in a SEPARATE THREAD with a fresh loop.
    import asyncio, threading
    try:
        asyncio.get_running_loop()   # raises RuntimeError if there is NO running loop (i.e. a plain script)
        in_loop = True               # we ARE inside a loop (Jupyter)
    except RuntimeError:
        in_loop = False              # no loop -> we are a plain script
    if not in_loop:
        return asyncio.run(coro)     # simple case: just run the coroutine

    box = {}                         # `box` = a dict to carry the result/error back out of the thread
    def worker():
        loop = asyncio.new_event_loop()    # a brand-new event loop for this thread (ProactorEventLoop on Windows)
        asyncio.set_event_loop(loop)       # make it the active loop for this thread
        try:
            box["value"] = loop.run_until_complete(coro)   # run the coroutine to completion, store its result
        except BaseException as exc:
            box["error"] = exc                              # capture any error to re-raise on the calling thread
        finally:
            loop.close()                                    # always clean up the loop
    t = threading.Thread(target=worker)   # `t` = the worker thread that owns the new loop
    t.start(); t.join()                   # start the thread and wait for it to finish
    if "error" in box:
        raise box["error"]                # if the worker raised, re-raise it here
    return box["value"]                   # otherwise return the coroutine's result

# Only run the LIVE agent if we have BOTH an API key AND all required packages installed:
if os.getenv("OPENAI_API_KEY") and have("mcp") and have("langgraph") and have("langchain_openai"):
    goal = (f"Assess {TARGET}: check its security headers, look up OWASP guidance for each finding, "
            "and give me a short risk report. Only use the provided tools.")   # `goal` = the agent's instructions
    print(run_async(run_live_agent(goal)))   # run the async agent (via our helper) and print its report
else:
    # Missing key/packages: explain what WOULD happen so the notebook still completes cleanly.
    print("Live agent skipped (needs OPENAI_API_KEY + mcp + langgraph + langchain-openai).")
    print("Set the key and `python recon_server_web.py` is launched automatically over stdio.")
    print("\nThe agent would: check_security_headers(TARGET) -> lookup_guidance(each finding)")
    print("-> grounded report, choosing each tool call itself instead of following a fixed script.")

# The Three Parts in One Table

- Stage 1 Chains: orchestration, data = LLM memory, fixed
- Stage 2 RAG: + real OWASP guidance, grounded, still fixed
- Stage 3 Agent+MCP: dynamic decisions, standardized recon tools
- Accuracy: medium → high → high
- Automation: low → medium → high

> ### Deeper Explanation
>
> Step back and see the arc whole, now on real data. Stage 1 gave us orchestration with chains, but facts came from the model's memory and the pipeline was fixed. Stage 2 kept the structure but grounded the findings in real OWASP guidance through RAG, trading guesses for citations. Stage 3 broke the fixed pipeline open: an agent decides the steps at runtime, and MCP gives it a standardized, swappable recon toolbox. The one-line story to leave students with: we start by using LLMs to interpret recon, then make them accurate with real guidance, and finally make them autonomous with agents and standardized tools.

In [ ]:
# `rows` = the comparison grid. Each tuple is (capability, Part 1, Part 2, Part 3).
rows = [
    ("Orchestration",    "Chains",     "Chains",            "Agents"),
    ("Data source",      "LLM memory", "RAG (OWASP KB)",    "RAG + MCP tools"),
    ("Accuracy",         "medium",     "high",              "high"),
    ("Automation",       "low",        "medium",            "high"),
    ("Flexibility",      "fixed",      "fixed",             "dynamic"),
    ("Tool integration", "manual",     "manual+retrieval",  "autonomous (agent+MCP)"),
]
print(f"  {'CAPABILITY':18s}{'PART 1':16s}{'PART 2':20s}{'PART 3'}")   # the header row
print("  " + "-" * 72)
for r in rows:                                  # print each row, padded into aligned columns
    print(f"  {r[0]:18s}{r[1]:16s}{r[2]:20s}{r[3]}")
print("\n  Story: interpret (chains) -> accurate (RAG) -> autonomous (agents + MCP).")

---
## Your Project — Take This to a Local Exploitable VM

This notebook used **non-intrusive HTTP recon** on authorized public targets so it is safe to run anywhere. Your project goes deeper, in an environment you fully control:

1. **Stand up a lab.** Install an intentionally-vulnerable VM (e.g., **Metasploitable 2/3** or OWASP **Juice Shop**) on an **isolated, host-only network**.
2. **Real scanning.** Run `nmap -sV` and other tools (e.g., `nikto`, `whatweb`, `gobuster`) against the VM — now you have *service* and *web* findings from a real scanner.
3. **Adapt the three stages.** Feed that real scan output through Stage 1 (chain report), Stage 2 (RAG over a CVE / OWASP knowledge base), and Stage 3 (an MCP server that wraps your tools so an agent can orchestrate them).
4. **Compare & reflect.** Show how grounding and autonomy change the output, and document the limitations you hit.

> **Rules of engagement.** Run scanners and exploits **only** against your own isolated lab VM or a target you are explicitly authorized to test. Keep the VM off the public network. Keep API keys in `.env` / Colab Secrets — never in code.